In [1]:
# 1) Download DENTEX and extract ONLY the disease folder (no GPU needed)
!pip -q install huggingface_hub scikit-image
from huggingface_hub import snapshot_download
snapshot_download(repo_id="keycxl/DENTEX", repo_type="dataset", local_dir="dentex_data")
!mkdir -p /content/dentex_train
!unzip -q /content/dentex_data/DENTEX/training_data.zip '*quadrant-enumeration-disease/*' -d /content/dentex_train
!find /content/dentex_train -name "*.json"
!ls /content/dentex_train/training_data/quadrant-enumeration-disease | head


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

/content/dentex_train/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json
train_quadrant_enumeration_disease.json
xrays


In [3]:
%%writefile train_classical.py

"""
Trains 6 traditional ML models on DENTEX disease data.

Training configuration:
- GroupShuffleSplit (test_size=0.3, random_state=42)
- StratifiedGroupKFold (3 folds)
- GridSearchCV (f1_macro)
- class_weight = balanced

Tuned hyperparameters:
- Logistic Regression: C=0.005, max_iter=150
- Ridge: alpha=1000, max_iter=50
- SVM: C=1, gamma=scale, kernel=rbf
- Decision Tree: criterion=entropy, max_depth=20,
                 min_samples_leaf=5, min_samples_split=2

Output:
models/classical/classical_models.joblib
"""

import glob
import os
import json
import sys
import joblib
import numpy as np
import pandas as pd
import cv2
import sklearn

from skimage.feature import hog

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold,
    GridSearchCV
)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)



ROOT = sys.argv[1] if len(sys.argv) > 1 else "/content"

js = glob.glob(
    f"{ROOT}/**/train_quadrant_enumeration_disease.json",
    recursive=True
)

assert js, "Disease JSON not found - pass the dataset folder."

data = json.load(open(js[0]))
folder = os.path.dirname(js[0])

paths = {
    os.path.basename(p): p
    for p in glob.glob(folder + "/**/*", recursive=True)
    if p.lower().endswith((".png", ".jpg"))
}

classes = {
    c["id"]: c["name"]
    for c in data["categories_3"]
}

imgs = {
    im["id"]: im
    for im in data["images"]
}

rows = [
    dict(
        image_id=a["image_id"],
        file=imgs[a["image_id"]]["file_name"],
        W=imgs[a["image_id"]]["width"],
        H=imgs[a["image_id"]]["height"],
        bbox=a["bbox"],
        label=a["category_id_3"]
    )
    for a in data["annotations"]
]

df = pd.DataFrame(rows).sort_values("file").reset_index(drop=True)

print("Total annotations:", len(df))
print("Disease classes:", classes)




clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

feats = []
cur = None
img = None

for r in df.itertuples():

    if r.file != cur:
        img = cv2.imread(
            paths[r.file],
            cv2.IMREAD_GRAYSCALE
        )
        cur = r.file

    x, y, w, h = [int(v) for v in r.bbox]

    x, y = max(x, 0), max(y, 0)

    crop = img[
        y:y + max(h, 1),
        x:x + max(w, 1)
    ]

    crop = cv2.resize(crop, (64, 64))

    crop = clahe.apply(crop)

    hog_features = hog(
        crop,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2)
    )

    bbox_features = [
        (x + w / 2) / r.W,
        (y + h / 2) / r.H,
        w / r.W,
        h / r.H
    ]

    feature_vector = np.concatenate([
        hog_features,
        bbox_features
    ])

    feats.append(feature_vector)


X = np.array(feats)

y = df["label"].values

g = df["image_id"].values

print("\nFeature matrix shape:", X.shape)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.3,
    random_state=42
)

tr, te = next(
    splitter.split(X, y, groups=g)
)

print("\nTraining samples:", len(tr))
print("Testing samples:", len(te))



models = {

    # Logistic Regression
    "logreg": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=0.005,
            max_iter=150,
            class_weight="balanced"
        )
    ),

    # Ridge Classifier
    "ridge": make_pipeline(
        StandardScaler(),
        RidgeClassifier(
            alpha=1000,
            max_iter=50,
            class_weight="balanced"
        )
    ),

    # Random Forest
    "rf": RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ),

    # Support Vector Machine
    "svm": make_pipeline(
        StandardScaler(),
        PCA(
            n_components=0.95,
            random_state=42
        ),
        SVC(
            C=1,
            gamma="scale",
            kernel="rbf",
            class_weight="balanced"
        )
    ),

    # K-Nearest Neighbours
    "knn": make_pipeline(
        StandardScaler(),
        PCA(random_state=42),
        KNeighborsClassifier()
    ),

    # Decision Tree
    "dt": DecisionTreeClassifier(
        criterion="entropy",
        max_depth=20,
        min_samples_leaf=5,
        min_samples_split=2,
        class_weight="balanced",
        random_state=42
    )
}



grids = {

    "logreg": {
        "logisticregression__C": [0.005]
    },

    "ridge": {
        "ridgeclassifier__alpha": [1000]
    },

    "rf": {
        "n_estimators": [200],
        "max_depth": [20]
    },

    "svm": {
        "svc__C": [1],
        "svc__gamma": ["scale"],
        "svc__kernel": ["rbf"]
    },

    "knn": {
        "pca__n_components": [0.90],
        "kneighborsclassifier__n_neighbors": [3]
    },

    "dt": {
        "criterion": ["entropy"],
        "max_depth": [20],
        "min_samples_leaf": [5],
        "min_samples_split": [2]
    }
}



cv = StratifiedGroupKFold(
    n_splits=3
)

best = {}
metrics = []

for n in models:

    print("\n" + "=" * 60)
    print("TRAINING MODEL:", n.upper())
    print("=" * 60)

    gs = GridSearchCV(
        estimator=models[n],
        param_grid=grids[n],
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    gs.fit(
        X[tr],
        y[tr],
        groups=g[tr]
    )

    p = gs.predict(X[te])

    best[n] = gs.best_estimator_

    accuracy = accuracy_score(y[te], p)

    precision = precision_score(
        y[te],
        p,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y[te],
        p,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y[te],
        p,
        average="macro",
        zero_division=0
    )

    metrics.append({

        "model": n,

        "accuracy": round(accuracy, 3),

        "precision": round(precision, 3),

        "recall": round(recall, 3),

        "f1": round(f1, 3),

        "params": str(gs.best_params_)

    })

    print("Best parameters:", gs.best_params_)

    print("Best CV F1:", round(gs.best_score_, 3))

    print("Test Accuracy:", round(accuracy, 3))

    print("Test Precision:", round(precision, 3))

    print("Test Recall:", round(recall, 3))

    print("Test F1:", round(f1, 3))


os.makedirs(
    "models/classical",
    exist_ok=True
)

joblib.dump(
    {
        "models": best,
        "classes": classes,
        "metrics": metrics,
        "sklearn": sklearn.__version__
    },
    "models/classical/classical_models.joblib"
)

print("\n" + "=" * 60)

print("ALL SIX MODELS TRAINED SUCCESSFULLY")

print("=" * 60)

print("\nFINAL MODEL COMPARISON")

results_df = pd.DataFrame(metrics)

print(results_df.to_string(index=False))

print("\nSaved to:")
print("models/classical/classical_models.joblib")

print("\nScikit-learn version:", sklearn.__version__)

Overwriting train_classical.py


In [4]:
!python train_classical.py /content/dentex_train


Total annotations: 3529
Disease classes: {0: 'Impacted', 1: 'Caries', 2: 'Periapical Lesion', 3: 'Deep Caries'}

Feature matrix shape: (3529, 1768)

Training samples: 2468
Testing samples: 1061

TRAINING MODEL: LOGREG
Best parameters: {'logisticregression__C': 0.005}
Best CV F1: 0.479
Test Accuracy: 0.609
Test Precision: 0.49
Test Recall: 0.543
Test F1: 0.505

TRAINING MODEL: RIDGE
Best parameters: {'ridgeclassifier__alpha': 1000}
Best CV F1: 0.473
Test Accuracy: 0.582
Test Precision: 0.472
Test Recall: 0.561
Test F1: 0.488

TRAINING MODEL: RF
Best parameters: {'max_depth': 20, 'n_estimators': 200}
Best CV F1: 0.344
Test Accuracy: 0.716
Test Precision: 0.401
Test Recall: 0.395
Test F1: 0.382

TRAINING MODEL: SVM
Best parameters: {'svc__C': 1, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
Best CV F1: 0.487
Test Accuracy: 0.701
Test Precision: 0.591
Test Recall: 0.519
Test F1: 0.505

TRAINING MODEL: KNN
Best parameters: {'kneighborsclassifier__n_neighbors': 3, 'pca__n_components': 0.9}
Be

In [5]:
# 4) Download the result
import sklearn
print("scikit-learn version used:", sklearn.__version__)
from google.colab import files
files.download("/content/models/classical/classical_models.joblib")


scikit-learn version used: 1.6.1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>